# 14. HuggingFace Transformers

**Цель:** Познакомиться с библиотекой HuggingFace Transformers: загрузить предобученные BERT и GPT-2, выполнить fine-tuning для классификации, сравнить с нашей реализацией.

---

In [1]:
# Импорты: стандартная библиотека, PyTorch, NumPy, Matplotlib
import sys, os, math
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
# Автоопределение устройства: CUDA > MPS > CPU
if torch.cuda.is_available():
    device = torch.device('cuda')
elif torch.backends.mps.is_available():
    device = torch.device('mps')
else:
    device = torch.device('cpu')



2026-06-05 23:18:02,611 [INFO] hf: Using device: cuda


In [2]:
# Импорты HuggingFace: токенизатор, модели, пайплайны, Trainer
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification, AutoModelForCausalLM,
    pipeline, Trainer, TrainingArguments, BertForSequenceClassification
)
from datasets import load_dataset



2026-06-05 23:18:03,719 [INFO] hf: Loading HuggingFace libraries
2026-06-05 23:18:06,273 [DEBUG] datasets: PyTorch version 2.11.0+cu128 available.
2026-06-05 23:18:06,558 [INFO] hf: HuggingFace libraries loaded


## 14.1 BERT: загрузка предобученной модели и токенизация

In [3]:
# distilbert — компактная версия BERT (быстрее, чуть хуже точность)
model_name = 'distilbert-base-uncased'
tokenizer = AutoTokenizer.from_pretrained(model_name)
bert_model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2).to(device)

# Демонстрация токенизации: BERT использует WordPiece (подсловные токены)
text = 'Transformers are amazing for natural language processing!'
tokens = tokenizer(text, return_tensors='pt').to(device)
print(f'Original: {text}')
print(f'Tokens:   {tokenizer.convert_ids_to_tokens(tokens["input_ids"][0])}')
print(f'Input IDs: {tokens["input_ids"][0].tolist()}')


2026-06-05 23:18:06,571 [INFO] hf: Loading BERT tokenizer and model (distilbert for speed)
2026-06-05 23:18:06,695 [DEBUG] httpcore.connection: connect_tcp.started host='huggingface.co' port=443 local_address=None timeout=10 socket_options=None
2026-06-05 23:18:06,814 [DEBUG] httpcore.connection: connect_tcp.complete return_value=<httpcore._backends.sync.SyncStream object at 0x0000016A12115400>
2026-06-05 23:18:06,815 [DEBUG] httpcore.connection: start_tls.started ssl_context=<ssl.SSLContext object at 0x0000016A3CE18390> server_hostname='huggingface.co' timeout=10
2026-06-05 23:18:06,845 [DEBUG] httpcore.connection: start_tls.complete return_value=<httpcore._backends.sync.SyncStream object at 0x0000016A3CD9F110>
2026-06-05 23:18:06,845 [DEBUG] httpcore.http11: send_request_headers.started request=<Request [b'HEAD']>
2026-06-05 23:18:06,846 [DEBUG] httpcore.http11: send_request_headers.complete
2026-06-05 23:18:06,846 [DEBUG] httpcore.http11: send_request_body.started request=<Request [

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
2026-06-05 23:18:08,340 [INFO] hf: BERT model loaded: distilbert-base-uncased, params=66955010


Original: Transformers are amazing for natural language processing!
Tokens:   ['[CLS]', 'transformers', 'are', 'amazing', 'for', 'natural', 'language', 'processing', '!', '[SEP]']
Input IDs: [101, 19081, 2024, 6429, 2005, 3019, 2653, 6364, 999, 102]


## 14.2 Fine-tuning BERT на IMDb

Обучим BERT на задаче классификации тональности (sentiment analysis).

In [5]:

# Загружаем подмножество IMDB: 100 тренировочных, 50 тестовых
dataset = load_dataset('stanfordnlp/imdb', split=['train[:100]', 'test[:50]'])
train_dataset, test_dataset = dataset

print(f'Train samples: {len(train_dataset)}')
print(f'Test samples:  {len(test_dataset)}')
print(f'Example: {train_dataset[0]["text"][:100]}...')
print(f'Label: {train_dataset[0]["label"]} (0=neg, 1=pos)')



2026-06-05 23:18:23,323 [INFO] hf: Loading IMDb dataset (small subset)
2026-06-05 23:18:23,324 [DEBUG] httpcore.connection: close.started
2026-06-05 23:18:23,325 [DEBUG] httpcore.connection: close.complete
2026-06-05 23:18:23,325 [DEBUG] httpcore.connection: close.started
2026-06-05 23:18:23,325 [DEBUG] httpcore.connection: close.complete
2026-06-05 23:18:23,326 [DEBUG] httpcore.connection: close.started
2026-06-05 23:18:23,326 [DEBUG] httpcore.connection: close.complete
2026-06-05 23:18:23,327 [DEBUG] httpcore.connection: connect_tcp.started host='huggingface.co' port=443 local_address=None timeout=10 socket_options=None
2026-06-05 23:18:23,366 [DEBUG] httpcore.connection: connect_tcp.complete return_value=<httpcore._backends.sync.SyncStream object at 0x0000016A3D42F460>
2026-06-05 23:18:23,367 [DEBUG] httpcore.connection: start_tls.started ssl_context=<ssl.SSLContext object at 0x0000016A3CE18390> server_hostname='huggingface.co' timeout=10
2026-06-05 23:18:23,406 [DEBUG] httpcore.con

Train samples: 100
Test samples:  50
Example: I rented I AM CURIOUS-YELLOW from my video store because of all the controversy that surrounded it w...
Label: 0 (0=neg, 1=pos)


In [6]:
# Функция токенизации: padding до макс. длины, обрезка длинных текстов
def tokenize_function(examples):
    return tokenizer(examples['text'], padding='max_length', truncation=True, max_length=256)

# .map() применяет функцию ко всем примерам в датасете (batched=True — быстрее)
train_enc = train_dataset.map(tokenize_function, batched=True)
test_enc = test_dataset.map(tokenize_function, batched=True)

# Trainer ожидает поле 'labels' (не 'label') — переименовываем
train_enc = train_enc.rename_column('label', 'labels')
test_enc = test_enc.rename_column('label', 'labels')
# Переключаем формат на torch.Tensor для совместимости с Trainer
train_enc.set_format(type='torch', columns=['input_ids', 'attention_mask', 'labels'])
test_enc.set_format(type='torch', columns=['input_ids', 'attention_mask', 'labels'])



2026-06-05 23:18:27,332 [DEBUG] hf: Tokenizing IMDb dataset
2026-06-05 23:18:27,466 [INFO] hf: IMDb tokenized: 100 train, 50 test


In [7]:
# TrainingArguments — гиперпараметры обучения
training_args = TrainingArguments(
    output_dir='./checkpoints/hf-bert-imdb',
    num_train_epochs=2,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    logging_steps=5,
    eval_strategy='steps',  # оценка каждые eval_steps шагов
    eval_steps=10,
    save_strategy='no',  # не сохраняем чекпоинты
    report_to='none',
    disable_tqdm=True,
)

# Trainer — высокоуровневое API: инкапсулирует цикл обучения
trainer = Trainer(
    model=bert_model,
    args=training_args,
    train_dataset=train_enc,
    eval_dataset=test_enc,
)

# Запуск: Trainer сам управляет forward/backward/optimizer
trainer.train()



2026-06-05 23:18:28,960 [INFO] hf: Fine-tuning BERT on IMDb (2 epochs)


{'loss': '0.5421', 'grad_norm': '2.965', 'learning_rate': '4.231e-05', 'epoch': '0.3846'}
{'loss': '0.1389', 'grad_norm': '1.188', 'learning_rate': '3.269e-05', 'epoch': '0.7692'}
{'eval_loss': '0.05882', 'eval_runtime': '0.0781', 'eval_samples_per_second': '639.8', 'eval_steps_per_second': '89.58', 'epoch': '0.7692'}
{'loss': '0.04337', 'grad_norm': '0.536', 'learning_rate': '2.308e-05', 'epoch': '1.154'}
{'loss': '0.0215', 'grad_norm': '0.3288', 'learning_rate': '1.346e-05', 'epoch': '1.538'}
{'eval_loss': '0.01679', 'eval_runtime': '0.0739', 'eval_samples_per_second': '676.5', 'eval_steps_per_second': '94.71', 'epoch': '1.538'}


2026-06-05 23:18:30,565 [INFO] hf: BERT fine-tuning complete


{'loss': '0.01576', 'grad_norm': '0.2697', 'learning_rate': '3.846e-06', 'epoch': '1.923'}
{'eval_loss': '0.01406', 'eval_runtime': '0.0836', 'eval_samples_per_second': '597.9', 'eval_steps_per_second': '83.71', 'epoch': '2'}
{'train_runtime': '1.407', 'train_samples_per_second': '142.1', 'train_steps_per_second': '18.48', 'train_loss': '0.1471', 'epoch': '2'}


In [8]:
# Оценка на тестовой выборке — возвращает dict с метриками
results = trainer.evaluate()
print(f'Evaluation loss: {results["eval_loss"]:.4f}')

# Ручная проверка на нескольких примерах для интуиции
test_texts = [
    'This movie was absolutely fantastic! I loved every minute.',
    'Terrible waste of time. The acting was horrible.',
    'It was okay, not great but not terrible either.',  # нейтральный — сложный случай
]
for text in test_texts:
    inputs = tokenizer(text, return_tensors='pt', padding=True, truncation=True).to(device)
    with torch.no_grad():
        outputs = bert_model(**inputs)
    pred = outputs.logits.argmax(-1).item()  # индекс класса с макс. вероятностью
    sentiment = 'positive' if pred == 1 else 'negative'
    print(f'  "{text[:50]}..." -> {sentiment}')



2026-06-05 23:18:36,531 [DEBUG] hf: Evaluating fine-tuned BERT
2026-06-05 23:18:36,668 [INFO] hf: BERT evaluation complete


{'eval_loss': '0.01406', 'eval_runtime': '0.0823', 'eval_samples_per_second': '607.4', 'eval_steps_per_second': '85.03', 'epoch': '2'}
Evaluation loss: 0.0141
  'This movie was absolutely fantastic! I loved every...' -> negative
  'Terrible waste of time. The acting was horrible....' -> negative
  'It was okay, not great but not terrible either....' -> negative


## 14.3 GPT-2: генерация текста с предобученной моделью

In [9]:
# Загружаем GPT-2 — авторегрессионная модель для генерации текста
gpt2_name = 'gpt2'
gpt2_tokenizer = AutoTokenizer.from_pretrained(gpt2_name)
gpt2_model = AutoModelForCausalLM.from_pretrained(gpt2_name).to(device)
gpt2_tokenizer.pad_token = gpt2_tokenizer.eos_token  # GPT-2 не имеет pad_token — используем eos

prompt = 'The future of artificial intelligence is'
inputs = gpt2_tokenizer(prompt, return_tensors='pt').to(device)

# Генерация с top-p (nucleus) сэмплингом и температурой
with torch.no_grad():
    outputs = gpt2_model.generate(
        **inputs,
        max_new_tokens=50,
        temperature=0.8,  # < 1 — более уверенные предсказания
        top_p=0.9,  # nucleus: выбираем из токенов с кумулятивной вероятностью 90%
        do_sample=True,  # стохастическая выборка (не greedy)
        pad_token_id=gpt2_tokenizer.eos_token_id,
    )

generated = gpt2_tokenizer.decode(outputs[0], skip_special_tokens=True)
print(f'Prompt: {prompt}')
print(f'Generated: {generated}')



2026-06-05 23:19:00,071 [INFO] hf: Loading GPT-2 for text generation
2026-06-05 23:19:00,073 [DEBUG] httpcore.connection: close.started
2026-06-05 23:19:00,073 [DEBUG] httpcore.connection: close.complete
2026-06-05 23:19:00,074 [DEBUG] httpcore.connection: close.started
2026-06-05 23:19:00,074 [DEBUG] httpcore.connection: close.complete
2026-06-05 23:19:00,074 [DEBUG] httpcore.connection: close.started
2026-06-05 23:19:00,075 [DEBUG] httpcore.connection: close.complete
2026-06-05 23:19:00,075 [DEBUG] httpcore.connection: connect_tcp.started host='huggingface.co' port=443 local_address=None timeout=10 socket_options=None
2026-06-05 23:19:00,101 [DEBUG] httpcore.connection: connect_tcp.complete return_value=<httpcore._backends.sync.SyncStream object at 0x0000016A11EA3070>
2026-06-05 23:19:00,101 [DEBUG] httpcore.connection: start_tls.started ssl_context=<ssl.SSLContext object at 0x0000016A3CE18390> server_hostname='huggingface.co' timeout=10
2026-06-05 23:19:00,126 [DEBUG] httpcore.conne

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

2026-06-05 23:19:01,494 [DEBUG] httpcore.http11: send_request_headers.started request=<Request [b'HEAD']>
2026-06-05 23:19:01,494 [DEBUG] httpcore.http11: send_request_headers.complete
2026-06-05 23:19:01,495 [DEBUG] httpcore.http11: send_request_body.started request=<Request [b'HEAD']>
2026-06-05 23:19:01,495 [DEBUG] httpcore.http11: send_request_body.complete
2026-06-05 23:19:01,495 [DEBUG] httpcore.http11: receive_response_headers.started request=<Request [b'HEAD']>
2026-06-05 23:19:01,690 [DEBUG] httpcore.http11: receive_response_headers.complete return_value=(b'HTTP/1.1', 200, b'OK', [(b'Content-Type', b'text/plain; charset=utf-8'), (b'Content-Length', b'124'), (b'Connection', b'keep-alive'), (b'Date', b'Fri, 05 Jun 2026 20:19:01 GMT'), (b'ETag', b'"3dc481ecc3b2c47a06ab4e20dba9d7f4b447bdf3"'), (b'X-Powered-By', b'huggingface-moon'), (b'X-Request-Id', b'Root=1-6a232f35-62e58ca22e8e027b5b0b1d11;44d8df47-71da-4a53-a8a0-4ae6bd30825b'), (b'RateLimit', b'"resolvers";r=2975;t=41'), (b'Ra

Prompt: The future of artificial intelligence is
Generated: The future of artificial intelligence is being discussed in the United States, as it looks to develop a new way of doing science.

"We are seeing an important step towards a new understanding of artificial intelligence, because we are seeing a significant number of people who are still not used


## 14.4 Pipeline API

HuggingFace Pipeline — высокоуровневый API для быстрого запуска моделей.

In [10]:
# Pipeline API — универсальный интерфейс (токенизация + модель + постобработка)
classifier = pipeline('text-classification', model=bert_model, tokenizer=tokenizer, device=0 if device.type in ('cuda', 'mps') else -1)

samples = [
    'I really enjoyed this film, great acting!',
    'This was boring and way too long.',
]
# Pipeline автоматически батчит и возвращает список result[{'label', 'score'}]
for text, result in zip(samples, classifier(samples)):
    print(f'  "{text[:40]}..." -> {result["label"]} (score: {result["score"]:.3f})')

# Тот же Pipeline для генерации текста — единый интерфейс
generator = pipeline('text-generation', model=gpt2_model, tokenizer=gpt2_tokenizer, device=0 if device.type in ('cuda', 'mps') else -1)
result = generator('Transformers have revolutionized', max_new_tokens=30, num_return_sequences=1)
print(f'  Generated: {result[0]["generated_text"]}')



2026-06-05 23:19:48,187 [DEBUG] hf: Using HuggingFace pipelines
[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'num_return_sequences'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=30) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


  'I really enjoyed this film, great acting...' -> LABEL_0 (score: 0.970)
  'This was boring and way too long....' -> LABEL_0 (score: 0.937)


[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer GPT2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.
2026-06-05 23:19:48,432 [INFO] hf: Pipeline API demo complete


  Generated: Transformers have revolutionized the way we think about technology.

The new era of technology comes from the likes of Microsoft and Google, which have been developing a suite of


## 14.5 Сравнение: наша реализация vs HuggingFace

| Аспект | Наша реализация | HuggingFace |
|--------|----------------|-------------|
| Размер | ~50K параметров | ~67M (BERT-base) |
| Данные | Синтетические | Wikipedia + BookCorpus |
| Время обучения | Минуты | Дни на TPU |
| Качество | Базовое | SOTA |
| Гибкость | Полный контроль | Высокоуровневый API |
| Скорость inference | Медленнее | Оптимизировано (kernel fusion) |

**Вывод:** Наша реализация помогла понять механизмы изнутри. HuggingFace — для production.

In [11]:
# Итоговый вывод с перечислением изученного в этом ноутбуке
print("=== HuggingFace Transformers complete ===")
print("Topics covered:")
print("  - Loading BERT (distilbert) with AutoTokenizer/AutoModel")
print("  - Fine-tuning BERT for text classification on IMDb")
print("  - Trainer API")
print("  - GPT-2 text generation (top-p, temperature)")
print("  - HuggingFace Pipeline API")
print("  - Comparison: our implementation vs HuggingFace")



2026-06-05 23:19:54,095 [INFO] hf: HuggingFace notebook complete


=== HuggingFace Transformers complete ===
Topics covered:
  - Loading BERT (distilbert) with AutoTokenizer/AutoModel
  - Fine-tuning BERT for text classification on IMDb
  - Trainer API
  - GPT-2 text generation (top-p, temperature)
  - HuggingFace Pipeline API
  - Comparison: our implementation vs HuggingFace
